# LeetCode 2525 — Categorize Box According to Criteria bằng Python

Link đề: https://leetcode.com/problems/categorize-box-according-to-criteria/ · Độ khó: **Easy** · Chủ đề: Math, Simulation (if/else)

Skeleton LeetCode:

```python
class Solution:
    def categorizeBox(self, length: int, width: int, height: int, mass: int) -> str:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: phân loại một cái hộp theo hai tiêu chí **Bulky** (cồng kềnh) và **Heavy** (nặng) thành một trong 4 nhãn `"Both"`, `"Bulky"`, `"Heavy"`, `"Neither"`.
2. Viết điều kiện logic với `>=`, `or`, `and`, và hiểu vì sao **biên** (bằng đúng `10^4`, `10^9`, `100`) quan trọng.
3. Dùng ba kỹ thuật "ánh xạ điều kiện → nhãn": chuỗi `if/elif`, tra bảng (list/dict) bằng cặp cờ boolean, và dựng nhãn từ danh sách.
4. Hiểu rủi ro **tràn số nguyên** ở ngôn ngữ khác (`length * width * height` tới `10^15`) và cách kiểm tra "an toàn" bằng phép chia; Python không tràn nhưng cần hiểu ý.
5. Tránh các lỗi: dùng `>` thay `>=`, dùng `and` thay `or` cho các cạnh, quên nhãn `"Both"`, viết `10 ^ 9` (XOR) thay vì `10 ** 9`.

## 2. Đề bài & ví dụ

Cho bốn số nguyên `length`, `width`, `height` (kích thước hộp) và `mass` (khối lượng). Phân loại hộp:

- Hộp là **Bulky** nếu **ít nhất một** trong các điều kiện sau đúng:
  - ít nhất một cạnh (`length`, `width` hoặc `height`) `>= 10^4`;
  - **thể tích** `length * width * height >= 10^9`.
- Hộp là **Heavy** nếu `mass >= 100`.

Trả về một chuỗi:

| Điều kiện | Nhãn |
|---|---|
| vừa Bulky vừa Heavy | `"Both"` |
| Bulky nhưng không Heavy | `"Bulky"` |
| Heavy nhưng không Bulky | `"Heavy"` |
| không Bulky và không Heavy | `"Neither"` |

| Ví dụ | length, width, height, mass | Output | Giải thích |
|---|---|---|---|
| 1 | `1000, 35, 700, 300` | `"Heavy"` | không cạnh nào `>= 10^4`; thể tích `24.500.000 < 10^9` nên không Bulky; `mass = 300 >= 100` nên Heavy |
| 2 | `200, 50, 800, 50` | `"Neither"` | không Bulky (thể tích `8.000.000`), không Heavy (`50 < 100`) |

Constraints: `1 <= length, width, height <= 10^5`; `1 <= mass <= 10^3`.

In [1]:
length, width, height, mass = 1000, 35, 700, 300
volume = length * width * height
print("thể tích =", volume)
print("có cạnh >= 10^4?", max(length, width, height) >= 10**4)
print("thể tích >= 10^9?", volume >= 10**9)
print("mass >= 100?", mass >= 100)

thể tích = 24500000
có cạnh >= 10^4? False
thể tích >= 10^9? False
mass >= 100? True


## 3. Kiến thức nền

### 3.1. Toán tử so sánh và biên `>=` / `>`

`a >= b` đúng khi `a` **lớn hơn hoặc bằng** `b`; `a > b` loại trường hợp bằng. Với bài này, đề viết "ít nhất `10^4`" và "ít nhất `100`" nên **bằng đúng** ngưỡng vẫn được tính. Kiểm tra các giá trị sát ngưỡng (`9999`, `10000`, `99`, `100`) là cách bắt lỗi biên tốt nhất.

In [2]:
for v in [9999, 10000, 10001]:
    print(v, "| >= 10000:", v >= 10000, "| > 10000:", v > 10000)

9999 | >= 10000: False | > 10000: False
10000 | >= 10000: True | > 10000: False
10001 | >= 10000: True | > 10000: True


### 3.2. `or`, `and`, `any`, `max` — gộp nhiều điều kiện

`A or B` đúng nếu **một trong hai** đúng; `A and B` đúng khi **cả hai** đúng. "Có ít nhất một cạnh `>= 10^4`" nghĩa là `length >= 10**4 or width >= 10**4 or height >= 10**4`, tương đương `max(length, width, height) >= 10**4`, hoặc `any(x >= 10**4 for x in dims)`. Đừng nhầm với `all(...)`/`and` (cả ba cạnh).

In [3]:
dims = (10000, 1, 1)
print(any(x >= 10**4 for x in dims), all(x >= 10**4 for x in dims), max(dims) >= 10**4)

True False True


### 3.3. Luỹ thừa `**` khác XOR `^`

Trong Python, `10 ** 4` là `10000` (luỹ thừa), còn `10 ^ 4` là **XOR bit** (`10 ^ 4 = 14`). Viết `10^9` như trong toán học là bẫy: `10 ^ 9 == 3`! Hãy viết `10 ** 9` hoặc `1_000_000_000` (gạch dưới chỉ để dễ đọc). Không dùng `1e9` vì đó là số thực (float).

In [4]:
print(10 ** 4, 10 ^ 4)
print(10 ** 9, 10 ^ 9, 1_000_000_000 == 10 ** 9)
print(type(1e9), type(10 ** 9))

10000 14
1000000000 3 True
<class 'float'> <class 'int'>


### 3.4. Boolean cũng là số: `True == 1`, `False == 0`

Trong Python `True + True == 2` và `bool` có thể dùng làm chỉ số: `["a", "b"][True]` là `"b"`. Ta lợi dụng để mã hoá cặp cờ `(bulky, heavy)` thành một số `0..3` bằng `2 * bulky + heavy`, rồi tra một list nhãn.

In [5]:
print(True + True, False + True, 2 * True + False)
print(["a", "b"][True])
labels = ["Neither", "Heavy", "Bulky", "Both"]
for bulky in (False, True):
    for heavy in (False, True):
        code = 2 * bulky + heavy
        print(f"bulky={bulky!s:5} heavy={heavy!s:5} -> code {code} -> {labels[code]}")

2 1 2
b
bulky=False heavy=False -> code 0 -> Neither
bulky=False heavy=True  -> code 1 -> Heavy
bulky=True  heavy=False -> code 2 -> Bulky
bulky=True  heavy=True  -> code 3 -> Both


### 3.5. Từ điển với khoá là tuple

Khoá của `dict` có thể là một **tuple** (bất biến): `{(True, True): "Both", ...}`. Tra bảng `table[(bulky, heavy)]` tránh viết chuỗi `if/elif` dài và dễ kiểm tra đủ 4 trường hợp.

In [6]:
table = {(True, True): "Both", (True, False): "Bulky", (False, True): "Heavy", (False, False): "Neither"}
print(table[(True, False)])
print(len(table))

Bulky
4


### 3.6. Tràn số nguyên ở ngôn ngữ khác và kiểm tra "an toàn"

Thể tích tối đa là `10^5 * 10^5 * 10^5 = 10^15`, vượt xa `int` 32 bit (`~2,1 * 10^9`) — ở C++/Java phải dùng `long long`/`long`. Cách tránh tràn: nếu **đã** có cạnh `>= 10^4` thì kết luận Bulky ngay; còn lại mọi cạnh `< 10^4`, `length * width < 10^8` vẫn vừa `int` 32 bit, và ta kiểm tra `length * width >= ceil(10^9 / height)`. Python không tràn nên chỉ cần biết ý này.

In [7]:
height = 700
need = (10**9 + height - 1) // height          # trần của 10^9 / height
print("ngưỡng length*width để thể tích >= 10^9:", need)
for lw in [need - 1, need]:
    print(lw, lw * height, lw * height >= 10**9, lw >= need)

ngưỡng length*width để thể tích >= 10^9: 1428572
1428571 999999700 False False
1428572 1000000400 True True


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Cứ làm đúng như đề: hai câu hỏi độc lập "có Bulky không?" và "có Heavy không?", rồi ghép hai câu trả lời thành nhãn.

Ví dụ 1 (`1000, 35, 700, 300`):

```
Bulky?  cạnh lớn nhất = 1000 < 10000 ; thể tích = 1000*35*700 = 24,500,000 < 10^9  -> không
Heavy?  mass = 300 >= 100                                                          -> có
(không Bulky, Heavy) -> "Heavy"
```

Bảng chân trị 4 ô:

```
            Heavy = có   Heavy = không
Bulky  có   "Both"       "Bulky"
Bulky  không "Heavy"      "Neither"
```

Điều khó nhất không phải thuật toán mà là **đọc kỹ đề**: ngưỡng bao gồm dấu bằng, các cạnh dùng "một trong", thể tích dùng tích ba cạnh, và `Both` là ưu tiên đầu tiên.

## 5. Cách 1 — Brute Force / baseline

Tính `bulky` và `heavy` thành hai biến boolean rồi viết **chuỗi `if / elif / else`**, kiểm tra `Both` trước tiên (vì nó thoả cả hai).

"Nút thắt" ở đây không phải tốc độ (luôn `O(1)`) mà là độ dễ sai: phải nhớ đúng thứ tự `if`, các toán tử `>=`, và `or`/`and`.

In [8]:
class SolutionIfElse:
    def categorizeBox(self, length, width, height, mass):
        bulky = (length >= 10**4 or width >= 10**4 or height >= 10**4
                 or length * width * height >= 10**9)
        heavy = mass >= 100
        if bulky and heavy:
            return "Both"
        elif bulky:
            return "Bulky"
        elif heavy:
            return "Heavy"
        else:
            return "Neither"

print(SolutionIfElse().categorizeBox(1000, 35, 700, 300))
print(SolutionIfElse().categorizeBox(200, 50, 800, 50))

Heavy
Neither


**Độ phức tạp:** Time `O(1)`, Space `O(1)` — chỉ vài phép so sánh và một phép nhân.

## 6. Ý tưởng thuật toán / bước đột phá

Hai tiêu chí **độc lập** → có đúng `2 * 2 = 4` tổ hợp, mỗi tổ hợp một nhãn. Thay vì rẽ nhánh lồng nhau, ta **mã hoá** cặp `(bulky, heavy)` thành một chỉ số rồi tra bảng: ít rẽ nhánh hơn, không thể quên tổ hợp nào.

**Quy tắc "nếu… thì…":**

- Nếu có cạnh `>= 10^4` **hoặc** thể tích `>= 10^9` → `bulky = True`.
- Nếu `mass >= 100` → `heavy = True`.
- Tra nhãn theo `(bulky, heavy)`: `(T,T)` → `Both`, `(T,F)` → `Bulky`, `(F,T)` → `Heavy`, `(F,F)` → `Neither`.

Invariant: mỗi hộp thuộc **đúng một** trong bốn ô của bảng chân trị; hai cờ được tính độc lập nên không ảnh hưởng nhau.

In [9]:
labels = ["Neither", "Heavy", "Bulky", "Both"]
def classify(length, width, height, mass):
    bulky = max(length, width, height) >= 10**4 or length * width * height >= 10**9
    heavy = mass >= 100
    return labels[2 * bulky + heavy]

print(classify(1000, 35, 700, 300), classify(200, 50, 800, 50), classify(10000, 1, 1, 100))

Heavy Neither Both


## 7. Các cách tiếp theo

Ta có 4 cách: (1) `if / elif / else`; (2) cờ boolean + chỉ số list; (3) `dict` với khoá tuple; (4) kiểm tra thể tích **an toàn tràn số** bằng phép chia + dựng nhãn từ danh sách.

### Cách 2 — Cờ boolean làm chỉ số list

`code = 2 * bulky + heavy` thuộc `0..3`; list `["Neither", "Heavy", "Bulky", "Both"]` đặt đúng thứ tự mã hoá. Rất gọn nhưng cần ghi chú rõ thứ tự.

In [10]:
class SolutionIndex:
    def categorizeBox(self, length, width, height, mass):
        bulky = max(length, width, height) >= 10**4 or length * width * height >= 10**9
        heavy = mass >= 100
        return ["Neither", "Heavy", "Bulky", "Both"][2 * bulky + heavy]

print(SolutionIndex().categorizeBox(1000, 35, 700, 300))
print(SolutionIndex().categorizeBox(200, 50, 800, 50))

Heavy
Neither


Độ phức tạp Cách 2: Time `O(1)`, Space `O(1)`.

### Cách 3 — `dict` với khoá tuple

Liệt kê rõ cả 4 tổ hợp trong một `dict`. Dễ đọc nhất khi có nhiều hơn hai tiêu chí hoặc nhãn không theo quy luật nào.

In [11]:
class SolutionDict:
    TABLE = {
        (True, True): "Both",
        (True, False): "Bulky",
        (False, True): "Heavy",
        (False, False): "Neither",
    }

    def categorizeBox(self, length, width, height, mass):
        bulky = any(x >= 10**4 for x in (length, width, height)) or length * width * height >= 10**9
        heavy = mass >= 100
        return self.TABLE[(bulky, heavy)]

print(SolutionDict().categorizeBox(1000, 35, 700, 300))
print(SolutionDict().categorizeBox(200, 50, 800, 50))

Heavy
Neither


Độ phức tạp Cách 3: Time `O(1)`, Space `O(1)` (bảng 4 phần tử cố định).

### Cách 4 — Thể tích an toàn tràn số + dựng nhãn từ danh sách

Hai ý mới: (a) **không nhân ba cạnh**: nếu có cạnh `>= 10^4` thì Bulky ngay; nếu không, mọi cạnh `< 10^4` và ta so `length * width` với `ceil(10^9 / height)` — mọi số trung gian nhỏ (vừa `int` 32 bit cho tích hai cạnh). (b) dựng nhãn bằng danh sách các "tag" đã đúng: hai tag → `"Both"`, một tag → chính tag đó, không tag → `"Neither"`.

In [12]:
class SolutionSafe:
    def categorizeBox(self, length, width, height, mass):
        if max(length, width, height) >= 10**4:
            bulky = True
        else:
            need = (10**9 + height - 1) // height          # trần của 10^9 / height
            bulky = length * width >= need                  # tương đương length*width*height >= 10^9
        tags = []
        if bulky:
            tags.append("Bulky")
        if mass >= 100:
            tags.append("Heavy")
        if len(tags) == 2:
            return "Both"
        return tags[0] if tags else "Neither"

print(SolutionSafe().categorizeBox(1000, 35, 700, 300))
print(SolutionSafe().categorizeBox(200, 50, 800, 50))

Heavy
Neither


Độ phức tạp Cách 4: Time `O(1)`, Space `O(1)` (danh sách tối đa 2 phần tử).

## 8. TRACE từng bước

Trace Cách 1 với ba hộp: ví dụ 1, ví dụ 2 và hộp sát biên `(10000, 1, 1, 100)`.

| hộp | max cạnh | thể tích | `bulky` | `mass >= 100` | nhãn |
|---|---|---|---|---|---|
| `1000, 35, 700, 300` | 1000 | 24.500.000 | `False` | `True` | `Heavy` |
| `200, 50, 800, 50` | 800 | 8.000.000 | `False` | `False` | `Neither` |
| `10000, 1, 1, 100` | 10000 (`>= 10^4`) | 10.000 | `True` | `True` | `Both` |

In [13]:
def trace_box(length, width, height, mass):
    big_side = max(length, width, height)
    volume = length * width * height
    by_side = big_side >= 10**4
    by_volume = volume >= 10**9
    bulky = by_side or by_volume
    heavy = mass >= 100
    print(f"hộp ({length}, {width}, {height}, mass={mass})")
    print(f"  cạnh lớn nhất {big_side} >= 10^4? {by_side} | thể tích {volume} >= 10^9? {by_volume} -> bulky={bulky}")
    print(f"  mass {mass} >= 100? {heavy} -> heavy={heavy}")
    label = ["Neither", "Heavy", "Bulky", "Both"][2 * bulky + heavy]
    print(f"  nhãn = {label}")
    return label

trace_box(1000, 35, 700, 300)
trace_box(200, 50, 800, 50)
trace_box(1000, 1000, 1000, 99)

hộp (1000, 35, 700, mass=300)
  cạnh lớn nhất 1000 >= 10^4? False | thể tích 24500000 >= 10^9? False -> bulky=False
  mass 300 >= 100? True -> heavy=True
  nhãn = Heavy
hộp (200, 50, 800, mass=50)
  cạnh lớn nhất 800 >= 10^4? False | thể tích 8000000 >= 10^9? False -> bulky=False
  mass 50 >= 100? False -> heavy=False
  nhãn = Neither
hộp (1000, 1000, 1000, mass=99)
  cạnh lớn nhất 1000 >= 10^4? False | thể tích 1000000000 >= 10^9? True -> bulky=True
  mass 99 >= 100? False -> heavy=False
  nhãn = Bulky


'Bulky'

## 9. Code nộp LeetCode

In [14]:
class Solution:
    def categorizeBox(self, length: int, width: int, height: int, mass: int) -> str:
        bulky = (length >= 10**4 or width >= 10**4 or height >= 10**4
                 or length * width * height >= 10**9)
        heavy = mass >= 100
        if bulky and heavy:
            return "Both"
        if bulky:
            return "Bulky"
        if heavy:
            return "Heavy"
        return "Neither"

print(Solution().categorizeBox(1000, 35, 700, 300), Solution().categorizeBox(200, 50, 800, 50))

Heavy Neither


## 10. Vì sao thuật toán đúng

- **Hai cờ đúng định nghĩa:** `bulky` đúng ⇔ ít nhất một cạnh `>= 10^4` **hoặc** thể tích `>= 10^9` (phép `or` biểu diễn "ít nhất một điều kiện"); `heavy` đúng ⇔ `mass >= 100`. Đây là sao chép nguyên văn định nghĩa của đề.
- **Phân hoạch đủ và rời nhau:** cặp `(bulky, heavy)` nhận đúng một trong 4 giá trị; mỗi giá trị ánh xạ tới đúng một nhãn theo bảng đề. Nên mọi hộp có đúng một đáp án.
- **Thứ tự `if`:** kiểm tra `bulky and heavy` trước nhất, nên khi cả hai đúng ta không "rơi" vào nhánh `Bulky` hay `Heavy` (nếu `elif bulky` đứng trước thì `Both` không bao giờ được trả về).
- **Cách 4 (an toàn tràn):** khi mọi cạnh `< 10^4`, `length * width * height >= 10^9` ⇔ `length * width >= 10^9 / height` ⇔ `length * width >= ceil(10^9 / height)` (vì vế trái là số nguyên, `a >= x` ⇔ `a >= ceil(x)`). Không cần nhân cả ba cạnh.
- **Python không tràn:** số nguyên có độ chính xác tùy ý nên `length * width * height` (tối đa `10^15`) tính chính xác; ở C++/Java phải dùng 64 bit (`long long`).

## 11. Phân tích Time / Space Complexity

Đầu vào luôn là bốn số, không phụ thuộc kích thước nào.

| Cách | Time | Space |
|---|---|---|
| 1. `if / elif / else` | `O(1)` | `O(1)` |
| 2. Cờ làm chỉ số list | `O(1)` | `O(1)` |
| 3. `dict` khoá tuple | `O(1)` | `O(1)` |
| 4. Kiểm tra an toàn tràn + tag | `O(1)` | `O(1)` |

Cả bốn cách đều `O(1)`; khác biệt duy nhất là độ dễ đọc, độ bền khi mở rộng tiêu chí, và an toàn tràn số trong ngôn ngữ có số nguyên cố định.

## 12. Bộ test

Test 1: ví dụ đề và các giá trị **sát biên** (`9999`/`10000`, thể tích `10^9 - 1`/`10^9`, `mass` `99`/`100`).

In [15]:
impls = [SolutionIfElse(), SolutionIndex(), SolutionDict(), SolutionSafe(), Solution()]
tests = [
    ((1000, 35, 700, 300), "Heavy"),
    ((200, 50, 800, 50), "Neither"),
    ((1, 1, 1, 1), "Neither"),                          # nhỏ nhất
    ((1, 1, 1, 100), "Heavy"),                          # mass đúng 100
    ((1, 1, 1, 99), "Neither"),                         # mass 99
    ((10000, 1, 1, 1), "Bulky"),                        # length đúng 10^4
    ((1, 10000, 1, 1), "Bulky"),                        # width đúng 10^4
    ((1, 1, 10000, 1), "Bulky"),                        # height đúng 10^4
    ((9999, 9999, 1, 1), "Neither"),                    # cạnh 9999, thể tích 99.980.001 < 10^9
    ((1000, 1000, 1000, 1), "Bulky"),                   # thể tích đúng 10^9
    ((999, 1001, 1000, 1), "Neither"),                  # 999.999.000 < 10^9 (sát biên)
    ((10000, 1, 1, 100), "Both"),                       # bắt lỗi thiếu Both
    ((1000, 1000, 1000, 1000), "Both"),                 # mass tối đa
    ((100000, 100000, 100000, 1000), "Both"),           # biên trên mọi tham số
    ((9999, 9999, 9999, 99), "Bulky"),                  # chỉ nhờ thể tích 9999^3 ~ 10^12
    ((9999, 1, 1, 100), "Heavy"),                       # 9999 < 10^4 (bắt lỗi dùng > hay 9999)
]
for (l, w, h, m), exp in tests:
    assert 1 <= l <= 10**5 and 1 <= w <= 10**5 and 1 <= h <= 10**5 and 1 <= m <= 10**3
    for im in impls:
        assert im.categorizeBox(l, w, h, m) == exp, (type(im).__name__, (l, w, h, m), exp)
print("tất cả test cố định OK:", len(tests))

tất cả test cố định OK: 16


Test 2: ngẫu nhiên có seed, đúng constraints (`1..10^5` cho cạnh, `1..10^3` cho `mass`); lấy mẫu có thiên vị về các giá trị sát biên; oracle độc lập dùng `fractions.Fraction`.

In [16]:
import random
from fractions import Fraction
random.seed(2525)

def oracle(l, w, h, m):
    bulky = any(Fraction(x) >= Fraction(10) ** 4 for x in (l, w, h)) or Fraction(l) * w * h >= Fraction(10) ** 9
    heavy = Fraction(m) >= 100
    if bulky and heavy:
        return "Both"
    if bulky:
        return "Bulky"
    if heavy:
        return "Heavy"
    return "Neither"

side_pool = [1, 2, 999, 1000, 1001, 9998, 9999, 10000, 10001, 99999, 100000]
mass_pool = [1, 99, 100, 101, 500, 999, 1000]
counts = {}
for _ in range(5000):
    pick = lambda: random.choice(side_pool) if random.random() < 0.5 else random.randint(1, 10**5)
    l, w, h = pick(), pick(), pick()
    m = random.choice(mass_pool) if random.random() < 0.5 else random.randint(1, 10**3)
    assert 1 <= min(l, w, h) and max(l, w, h) <= 10**5 and 1 <= m <= 10**3
    exp = oracle(l, w, h, m)
    counts[exp] = counts.get(exp, 0) + 1
    for im in impls:
        assert im.categorizeBox(l, w, h, m) == exp, (type(im).__name__, (l, w, h, m))
print("5000 test ngẫu nhiên OK; phân bố nhãn:", counts)

5000 test ngẫu nhiên OK; phân bố nhãn: {'Both': 3948, 'Neither': 36, 'Bulky': 906, 'Heavy': 110}


Test 3: tính chất — kết quả luôn là một trong 4 nhãn; hoán vị ba cạnh không đổi nhãn; tăng `mass` (cố định kích thước) chỉ có thể chuyển `Neither → Heavy` hoặc `Bulky → Both`.

In [17]:
from itertools import permutations
random.seed(25)
valid = {"Both", "Bulky", "Heavy", "Neither"}
for _ in range(1000):
    l, w, h = (random.randint(1, 10**5) for _ in range(3))
    m = random.randint(1, 10**3)
    base = Solution().categorizeBox(l, w, h, m)
    assert base in valid
    for p in permutations((l, w, h)):
        assert Solution().categorizeBox(*p, m) == base
    heavier = Solution().categorizeBox(l, w, h, 1000)
    if base in ("Heavy", "Both"):
        assert heavier == base
    elif base == "Neither":
        assert heavier == "Heavy"
    else:
        assert heavier == "Both"
print("tính chất (4 nhãn, bất biến khi hoán vị cạnh, tăng mass): OK")

tính chất (4 nhãn, bất biến khi hoán vị cạnh, tăng mass): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: dùng `>` thay `>=` (sai ở biên)

In [18]:
def wrong_strict(length, width, height, mass):
    bulky = length > 10**4 or width > 10**4 or height > 10**4 or length * width * height > 10**9   # SAI: phải >=
    heavy = mass > 100                                                                              # SAI: phải >=
    if bulky and heavy:
        return "Both"
    if bulky:
        return "Bulky"
    if heavy:
        return "Heavy"
    return "Neither"

args = (10000, 1, 1, 100)
print("sai :", wrong_strict(*args))
print("đúng:", Solution().categorizeBox(*args))
assert wrong_strict(*args) != Solution().categorizeBox(*args)

sai : Neither
đúng: Both


Vì sao sai: đề nói "**ít nhất** `10^4`" và "**ít nhất** `100`", nên giá trị **bằng** ngưỡng cũng tính. Với `(10000, 1, 1, 100)` đáp án đúng là `"Both"` nhưng bản sai ra `"Neither"`.

### Lỗi 2: dùng `and` thay cho `or` giữa các cạnh

In [19]:
def wrong_and(length, width, height, mass):
    bulky = (length >= 10**4 and width >= 10**4 and height >= 10**4) or length * width * height >= 10**9   # SAI
    heavy = mass >= 100
    if bulky and heavy:
        return "Both"
    if bulky:
        return "Bulky"
    if heavy:
        return "Heavy"
    return "Neither"

args = (10000, 1, 1, 1)
print("sai :", wrong_and(*args))
print("đúng:", Solution().categorizeBox(*args))
assert wrong_and(*args) != Solution().categorizeBox(*args)

sai : Neither
đúng: Bulky


Vì sao sai: đề chỉ cần **một** cạnh `>= 10^4` ("ít nhất một cạnh"). `and` đòi cả ba cạnh. **Sửa:** `or` (hoặc `max(...) >= 10**4`).

### Lỗi 3: quên nhãn `Both` (rẽ nhánh sai thứ tự)

In [20]:
def wrong_no_both(length, width, height, mass):
    bulky = max(length, width, height) >= 10**4 or length * width * height >= 10**9
    heavy = mass >= 100
    if bulky:                       # SAI: gặp bulky là trả ngay, bỏ qua trường hợp heavy
        return "Bulky"
    if heavy:
        return "Heavy"
    return "Neither"

args = (10000, 1, 1, 100)
print("sai :", wrong_no_both(*args))
print("đúng:", Solution().categorizeBox(*args))
assert wrong_no_both(*args) != Solution().categorizeBox(*args)

sai : Bulky
đúng: Both


Vì sao sai: khi vừa Bulky vừa Heavy phải trả `"Both"`. Hãy kiểm tra trường hợp "cả hai" **trước tiên** (hoặc tra bảng 4 ô).

### Lỗi 4: viết `10 ^ 9` thay cho `10 ** 9`

In [21]:
def wrong_xor(length, width, height, mass):
    bulky = max(length, width, height) >= 10 ** 4 or length * width * height >= 10 ^ 9     # SAI: ^ là XOR, 10 ^ 9 == 3
    heavy = mass >= 100
    if bulky and heavy:
        return "Both"
    if bulky:
        return "Bulky"
    if heavy:
        return "Heavy"
    return "Neither"

args = (1, 1, 4, 1)
print("10 ^ 9 =", 10 ^ 9)
print("sai :", wrong_xor(*args))
print("đúng:", Solution().categorizeBox(*args))
assert wrong_xor(*args) != Solution().categorizeBox(*args)

10 ^ 9 = 3
sai : Bulky
đúng: Neither


Vì sao sai: trong Python `^` là XOR theo bit, `10 ^ 9 == 3`, nên mọi hộp có thể tích `>= 3` bị coi là Bulky. Luỹ thừa phải viết `10 ** 9`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Bài "phân loại theo hai (hoặc nhiều) tiêu chí" → tính từng cờ boolean độc lập rồi ghép: bảng `2 x 2` hoặc `if bulky and heavy` đặt **đầu tiên**.
- Đọc kỹ "ít nhất" / "nhiều hơn": "ít nhất `x`" là `>= x`, "nhiều hơn `x`" là `> x`. Luôn test giá trị **bằng đúng** ngưỡng và sát ngưỡng ±1.
- "Ít nhất một cạnh/phần tử thoả" → `or` hoặc `any(...)`/`max(...)`; "mọi phần tử thoả" → `and` hoặc `all(...)`.
- Luỹ thừa trong Python là `**`; `^` là XOR. Tránh `1e9` (float); dùng `10**9` hoặc `1_000_000_000`.
- Ở C++/Java chú ý thể tích tới `10^15`: dùng `long long`/`long` hoặc so sánh bằng phép chia.
- Dùng `True`/`False` như `1`/`0` để mã hoá tổ hợp cờ thành chỉ số bảng.

## 15. Biến thể của bài

1. Thêm tiêu chí thứ ba (ví dụ **Fragile**) → 8 nhãn; khi đó bảng `dict` hoặc mã hoá bit `4*a + 2*b + c` gọn hơn `if/elif`.
2. Đổi ngưỡng: Bulky khi **tổng** ba cạnh `>= 10^4`, hoặc khi thể tích `> 10^9` (đổi `>=` thành `>`).
3. Cho **nhiều hộp** (mảng) và đếm số hộp mỗi nhãn bằng `Counter`.
4. Cho kích thước tới `10^9`: thể tích tới `10^27`; ngôn ngữ có số nguyên cố định phải dùng phép chia hoặc 128 bit.
5. Trả về **danh sách các nhãn áp dụng** (`["Bulky", "Heavy"]`) thay vì một chuỗi duy nhất.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. `if / elif / else` | rẽ nhánh theo thứ tự `Both` → `Bulky` → `Heavy` | `O(1)` | `O(1)` | **dễ đọc nhất**, nên dùng khi nộp |
| 2. Cờ làm chỉ số list | `2 * bulky + heavy` tra list | `O(1)` | `O(1)` | code ngắn, khi quy ước mã hoá rõ |
| 3. `dict` khoá tuple | liệt kê cả 4 tổ hợp | `O(1)` | `O(1)` | nhiều tiêu chí, nhãn bất quy tắc |
| 4. An toàn tràn + tag | chia thay vì nhân ba cạnh | `O(1)` | `O(1)` | ngôn ngữ có số nguyên cố định |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `categorizeBox(9999, 100, 100, 100)` trả về gì? (Gợi ý: kiểm tra thể tích.)

**Bài 2 (tính tay):** hộp `(2000, 2000, 300, 99)`: tính thể tích, cho biết có Bulky/Heavy không, nhãn là gì.

**Bài 3 (điền TODO):** hoàn thành `is_bulky(l, w, h)` trả `True`/`False` theo đúng định nghĩa Bulky.

**Bài 4 (tự viết lại):** viết `count_labels(boxes)` nhận danh sách bộ `(l, w, h, m)` và trả dict đếm số hộp mỗi nhãn.

**Bài 5 (mini challenge):** viết `categorize_three(l, w, h, m, fragile)` thêm tiêu chí `fragile` (bool) → trả nhãn dạng `"Bulky+Heavy+Fragile"` (nối các tag đúng bằng dấu `+`, hoặc `"Neither"` nếu không tag nào).

In [22]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().categorizeBox(9999, 100, 100, 100))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def is_bulky(l, w, h):
    # TODO: trả về True nếu có cạnh >= 10**4 hoặc l * w * h >= 10**9
    return False

Heavy


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** hai câu hỏi độc lập "Bulky?" và "Heavy?"; nhớ ngưỡng gồm dấu bằng.

**Mức 2 (vừa):** Bài 3: `return max(l, w, h) >= 10**4 or l * w * h >= 10**9`. Bài 4: dùng `Counter` hoặc `dict.get(label, 0) + 1`. Bài 5: dựng list tag bằng `if`, nối bằng `"+".join(tags)`.

**Mức 3 (gần đáp án):** Bài 1: cạnh lớn nhất `9999 < 10^4`; thể tích `9999 * 100 * 100 = 99.990.000 < 10^9` → không Bulky; `mass = 100 >= 100` → Heavy → `"Heavy"`. Bài 2: thể tích `2000 * 2000 * 300 = 1.200.000.000 >= 10^9` → Bulky; `99 < 100` → không Heavy → `"Bulky"`.

## 19. Đáp án tham khảo cho bài tập

In [23]:
# Bài 1: 9999*100*100 = 99.990.000 < 10^9, mass 100 -> Heavy
assert 9999 * 100 * 100 == 99_990_000
assert Solution().categorizeBox(9999, 100, 100, 100) == "Heavy"

# Bài 2: 2000*2000*300 = 1.2 * 10^9 -> Bulky ; mass 99 -> không Heavy
assert 2000 * 2000 * 300 == 1_200_000_000
assert Solution().categorizeBox(2000, 2000, 300, 99) == "Bulky"

# Bài 3
def is_bulky(l, w, h):
    return max(l, w, h) >= 10**4 or l * w * h >= 10**9
assert is_bulky(10000, 1, 1) and is_bulky(1000, 1000, 1000)
assert not is_bulky(9999, 1, 1) and not is_bulky(999, 1001, 1000)

# Bài 4
def count_labels(boxes):
    result = {}
    for l, w, h, m in boxes:
        label = Solution().categorizeBox(l, w, h, m)
        result[label] = result.get(label, 0) + 1
    return result
assert count_labels([(1000, 35, 700, 300), (200, 50, 800, 50), (1, 1, 1, 1)]) == {"Heavy": 1, "Neither": 2}
assert count_labels([(10000, 1, 1, 100)]) == {"Both": 1}
assert count_labels([]) == {}

# Bài 5
def categorize_three(l, w, h, m, fragile):
    tags = []
    if is_bulky(l, w, h):
        tags.append("Bulky")
    if m >= 100:
        tags.append("Heavy")
    if fragile:
        tags.append("Fragile")
    return "+".join(tags) if tags else "Neither"
assert categorize_three(10000, 1, 1, 100, True) == "Bulky+Heavy+Fragile"
assert categorize_three(1, 1, 1, 1, False) == "Neither"
assert categorize_three(1, 1, 1, 100, True) == "Heavy+Fragile"
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "phân loại theo hai cờ":**

```python
bulky = (length >= 10**4 or width >= 10**4 or height >= 10**4
         or length * width * height >= 10**9)
heavy = mass >= 100
if bulky and heavy:
    return "Both"
if bulky:
    return "Bulky"
if heavy:
    return "Heavy"
return "Neither"
```

**Checklist trước khi submit:**

- [ ] Ngưỡng dùng `>=` (ít nhất), không phải `>`?
- [ ] "Ít nhất một cạnh" dùng `or`/`max`, không phải `and`?
- [ ] Kiểm tra `Both` (cả hai) **trước** khi xét từng nhãn đơn?
- [ ] Luỹ thừa viết `10 ** 9` (không phải `10 ^ 9`, không phải `1e9`)?
- [ ] Đã thử các giá trị sát biên `9999`, `10000`, thể tích `10^9 - 1` và `10^9`, `mass` `99`/`100`?
- [ ] Ngôn ngữ khác: dùng `long long`/`long` cho thể tích?
- [ ] Chữ ký đúng `categorizeBox(self, length: int, width: int, height: int, mass: int) -> str`?